# Viewing MUR layers on the MAAP mapThe pipeline writes cloud-optimized GeoTIFFs beside each granule, and MAAP'stitiler tiles any COG by URL. That needs **no STAC registration, no pullrequest and no data-team approval** — the layers below are drawn straight fromyour workspace bucket.Run this in the ADE. It falls back to plain `ipyleaflet` if `stac_ipyleaflet`is not installed, so it works either way.

In [ ]:
import json, boto3, requestsWORKSPACE = "s3://maap-ops-workspace/jleach_jpl"TITILER   = "https://titiler-pgstac.maap-project.org"# Which day to draw. The pipeline writes one STAC item per day per mode.YEAR, DOY, MODE = 2026, 271, "nrt"item_key = f"mur/stac/items/{YEAR}/{DOY:03d}{MODE}.json"bucket   = WORKSPACE.split("/")[2]prefix   = "/".join(WORKSPACE.split("/")[3:])s3 = boto3.client("s3")body = s3.get_object(Bucket=bucket, Key=f"{prefix}/{item_key}")["Body"].read()item = json.loads(body)print(item["id"], "-", item["properties"]["datetime"])for k, a in item["assets"].items():    mark = "MAP" if "href_tilejson" in a else "   "    print(f"  {mark}  {k:15} {a['href'].rsplit('/', 1)[-1]}")

Assets marked `MAP` carry a ready-made TileJSON URL — the pipeline put it thereso the catalogue entry is enough on its own, without you needing to know whichtiler MAAP runs or how to address it.Each also carries an explicit `rescale`. That matters more than the colormap:titiler otherwise stretches every tile to its own data range, which produces aplausible-looking SST field whose colours mean something different each day.

In [ ]:
def tile_url(asset):    """Resolve an asset's TileJSON to the XYZ template a map layer wants."""    tj = requests.get(asset["href_tilejson"], timeout=60)    tj.raise_for_status()    return tj.json()["tiles"][0]drawable = {k: a for k, a in item["assets"].items() if "href_tilejson" in a}tiles = {}for k, a in drawable.items():    try:        tiles[k] = tile_url(a)        print(f"  ok    {k}")    except Exception as exc:        # Most often the COG is simply not there -- Stage 10b is non-fatal, so        # a field can be missing without the run having failed.        print(f"  SKIP  {k}: {exc}")

In [ ]:
try:    from stac_ipyleaflet import StacIpyleaflet    m = StacIpyleaflet(center=(20, -40), zoom=3)    add = lambda url, name: m.add_tile_layer(url=url, name=name)except ImportError:    from ipyleaflet import Map, TileLayer    m = Map(center=(20, -40), zoom=3)    add = lambda url, name: m.add_layer(TileLayer(url=url, name=name))for k, url in tiles.items():    add(url, k)m

## If a layer is blankCheck the granule's analysis level first — a capped run is structurallyidentical to a full one, so nothing on screen distinguishes them:```bashncdump -h <granule>.nc | grep mrva_analysis_level````mrva_analysis_level = 10` means the field is one doubling coarser thanproduction in each direction. It is smoother, not wrong.Then check titiler can see the object. It reads the workspace bucketdirectly, so a 404 here means the COG was never written:```pythonrequests.get(f"{TITILER}/cog/info", params={"url": item["assets"]["browse_sst"]["href"]}).json()```